In [7]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/5205 Project Proposal")
RAW_DIR = BASE_DIR / "Raw Dataset_RE"

keep_cols = ["Address", "Sale Price", "Property", "Sub-Nbhood", "Beds", "Bath", "Sale Date"]

remove_labels = ["average", "median", "totals"]

xlsx_files = sorted(RAW_DIR.rglob("*.xlsx"))
print(f"Found {len(xlsx_files)} files")

dfs = []

for fp in xlsx_files:
    try:
        df = pd.read_excel(fp)

        df.columns = df.columns.str.strip()
        cols_present = [c for c in keep_cols if c in df.columns]
        df = df[cols_present].copy()

        if "Address" in df.columns:
            df["Address"] = df["Address"].astype(str).str.strip()
            df = df[~df["Address"].str.lower().isin(remove_labels)]

        df = df.dropna(how="all")

        dfs.append(df)

    except Exception as e:
        print(f"Skipping {fp}: {e}")

combined = pd.concat(dfs, ignore_index=True)
combined["Sale Price"] = pd.to_numeric(combined["Sale Price"], errors="coerce")
combined["Sale Date"] = pd.to_datetime(combined["Sale Date"], errors="coerce")

print(f"Before: {combined.shape}")
combined["Address"] = combined["Address"].astype(str).str.strip()

combined = combined[
    (combined["Address"].notna()) &
    (combined["Address"].str.lower() != "nan") &
    (combined["Address"] != "")
]

print(f"After: {combined.shape}")

Found 120 files
Before: (23940, 6)
After: (23820, 6)


In [9]:
# df = pd.read_csv("/content/drive/MyDrive/5205 Project Proposal/cleaned_data.csv")
# df.head()

In [10]:
output_path = BASE_DIR / "map.csv"
combined["Sub-Nbhood"].value_counts().to_csv(output_path)

In [11]:
output_path = BASE_DIR / "cleaned_data.csv"
combined.to_csv(output_path, index=False)

## zipcode mapping: full and primary

In [12]:
src_path = "/content/drive/MyDrive/5205 Project Proposal/map.csv"
nb = pd.read_csv(src_path)["Sub-Nbhood"].astype(str).str.strip().tolist()

ZIP_FULL = {
    "Upper West Side": ["10023", "10024", "10025", "10069"],
    "Lincoln Square": ["10023", "10069"],
    "Upper East Side": ["10021", "10028", "10065", "10075", "10128"],
    "Yorkville": ["10028", "10075", "10128"],
    "Lenox Hill": ["10021", "10028", "10065"],
    "Chelsea": ["10001", "10011"],
    "Greenwich Village": ["10011", "10012", "10014"],
    "Murray Hill": ["10016", "10017"],
    "Hells Kitchen": ["10019", "10036", "10018"],
    "TriBeCa": ["10007", "10013"],
    "Turtle Bay": ["10017", "10022"],
    "Sutton Place": ["10022"],
    "Gramercy Park": ["10010", "10016"],
    "Midtown Center": ["10019", "10020", "10036", "10017", "10022"],
    "West Village": ["10014"],
    "Financial District": ["10004", "10005", "10006", "10038"],
    "Carnegie Hill": ["10021", "10028", "10128"],
    "East Village": ["10003", "10009"],
    "Lower East Side": ["10002", "10009"],
    "Kips Bay": ["10010", "10016"],
    "South Harlem": ["10026", "10027", "10030"],
    "Manhattan Valley": ["10025"],
    "Central Harlem": ["10026", "10027", "10030", "10037", "10039"],
    "Flatiron": ["10010", "10011", "10001"],
    "Battery Park City": ["10280", "10004"],
    "East Harlem": ["10029", "10035"],
    "NoMad": ["10001", "10010", "10016"],
    "Hudson Heights": ["10033", "10040"],
    "Morningside Heights": ["10025", "10027"],
    "SoHo": ["10012", "10013"],
    "Hamilton Heights": ["10031", "10032"],
    "Washington Heights": ["10032", "10033", "10040"],
    "Hudson Square": ["10013", "10014"],
    "South Street Seaport": ["10038"],
    "NoHo": ["10012", "10003"],
    "Midtown South": ["10001", "10016", "10018"],
    "Two Bridges": ["10002"],
    "Inwood": ["10034", "10040"],
    "Hudson Yards": ["10001", "10018"],
    "Roosevelt Island": ["10044"],
    "NoLita": ["10012", "10013"],
    "Chinatown": ["10013", "10002"],
    "Civic Center": ["10007", "10038"],
    "Fort George": ["10033", "10040"],
    "West Harlem & Manhattanville": ["10027", "10031"],
    "Little Italy": ["10012", "10013"],
}

missing = [s for s in nb if s not in ZIP_FULL]
if missing:
    raise ValueError(f"Missing ZIP mappings for: {missing}")

zip_full_df = pd.DataFrame({
    "Sub-Nbhood": nb,
    "ZIP_LIST": ["|".join(ZIP_FULL[s]) for s in nb],
    "ZIP_COUNT": [len(ZIP_FULL[s]) for s in nb],
})

out_path = BASE_DIR / "full_zip.csv"
zip_full_df.to_csv(out_path, index=False)

In [13]:
input_path = "/content/drive/MyDrive/5205 Project Proposal/map.csv"
nb_df = pd.read_csv(input_path)

nb_df["Sub-Nbhood"] = nb_df["Sub-Nbhood"].astype(str).str.strip()
sub_nbhoods = nb_df["Sub-Nbhood"].unique()

ZIP_PRIMARY = {
    "Upper West Side": "10024",
    "Lincoln Square": "10023",
    "Upper East Side": "10028",
    "Yorkville": "10128",
    "Lenox Hill": "10065",
    "Chelsea": "10011",
    "Greenwich Village": "10014",
    "Murray Hill": "10016",
    "Hells Kitchen": "10019",
    "TriBeCa": "10013",
    "Turtle Bay": "10017",
    "Sutton Place": "10022",
    "Gramercy Park": "10010",
    "Midtown Center": "10036",
    "West Village": "10014",
    "Financial District": "10005",
    "Carnegie Hill": "10128",
    "East Village": "10009",
    "Lower East Side": "10002",
    "Kips Bay": "10016",
    "South Harlem": "10026",
    "Manhattan Valley": "10025",
    "Central Harlem": "10027",
    "Flatiron": "10010",
    "Battery Park City": "10280",
    "East Harlem": "10029",
    "NoMad": "10016",
    "Hudson Heights": "10033",
    "Morningside Heights": "10025",
    "SoHo": "10012",
    "Hamilton Heights": "10031",
    "Washington Heights": "10033",
    "Hudson Square": "10013",
    "South Street Seaport": "10038",
    "NoHo": "10012",
    "Midtown South": "10001",
    "Two Bridges": "10002",
    "Inwood": "10034",
    "Hudson Yards": "10001",
    "Roosevelt Island": "10044",
    "NoLita": "10012",
    "Chinatown": "10013",
    "Civic Center": "10007",
    "Fort George": "10040",
    "West Harlem & Manhattanville": "10027",
    "Little Italy": "10013",
}

missing = [n for n in sub_nbhoods if n not in ZIP_PRIMARY]
if missing:
    print("Warning: Missing ZIP for:", missing)

zipcode_df = pd.DataFrame({
    "Sub-Nbhood": sub_nbhoods,
    "ZIP": [ZIP_PRIMARY.get(n) for n in sub_nbhoods]
})

out_path = BASE_DIR / "primary_zip.csv"
zipcode_df.to_csv(out_path, index=False)